**Importing Dependencies**

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_csv("insurance.csv")

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
46,42,83.0,1.57,25.57,True,Kolkata,unemployed,High
39,51,100.6,1.68,11.99,True,Bangalore,unemployed,High
30,35,89.6,1.73,32.97,False,Delhi,business_owner,Low
33,73,67.5,1.76,1.46,False,Mumbai,retired,Medium
70,69,99.9,1.65,0.57,False,Chandigarh,retired,High


In [4]:
df_feat = df.copy()

**Feature Engineering**

In [6]:
# 1st Feature
df_feat['bmi'] = df_feat['weight']/(df_feat['height']**2)

In [8]:
# 2nd Feature
def age_grp(age):
    if age < 25:
        return 'young'
    elif age < 45:
        return 'adult'
    elif age < 60:
        return 'middle aged'
    
    return 'senior'

In [9]:
df_feat['age_group'] = df_feat['age'].apply(age_grp)

In [10]:
# 3rd Feature
def lifestyle_risk(row):
    if row['smoker'] and row['bmi'] > 30:
        return 'High Risk'
    elif row['smoker'] and row['bmi'] > 27:
        return 'Medium Risk'
    else:
        return 'Low Risk'

In [12]:
df_feat['Life_style_risk'] = df_feat.apply(lifestyle_risk, axis=1)

In [13]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:
def tier_city(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    return 3

In [15]:
df_feat['city_tier'] = df_feat['city'].apply(tier_city)

In [23]:
df_feat = df_feat.drop(columns=['age', 'height', 'weight', 'smoker', 'city'])

**Selecting Features and Targets**

In [24]:
X = df_feat.drop(columns='insurance_premium_category')
Y = df_feat['insurance_premium_category']

In [25]:
X

,income_lpa,occupation,bmi,age_group,Life_style_risk,city_tier
0,2.92000,retired,49.227482,senior,Low Risk,2
1,34.28000,freelancer,30.189017,adult,Low Risk,1
2,36.64000,freelancer,21.118382,adult,Low Risk,2
3,3.34000,student,45.535900,young,High Risk,1
4,3.94000,retired,24.296875,senior,Low Risk,2
...,...,...,...,...,...,...
95,19.64000,business_owner,21.420747,adult,Low Risk,2
96,34.01000,private_job,47.984483,adult,Low Risk,1
97,44.86000,freelancer,18.765432,middle aged,Low Risk,1
98,28.30000,business_owner,30.521676,adult,Low Risk,1
